In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
## delete /content/nlp_project , be careful:
!rm -rf /content/nlp_project

In [ ]:
!git clone https://github.com/ethan-colin/generality-is-not-free
%cd generality-is-not-free

In [ ]:
# install dependencies in "requirements.txt"
!pip install -r requirements.txt

In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected")

PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [ ]:
!nvidia-smi

Fri Sep  4 13:34:06 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import transformers

print(transformers.__version__)

5.16.1


In [ ]:
# bringing training set into unified format
!python prepare_unified_data.py --drop data/drop_dataset_train.json --quoref data/quoref-train-v0.1.json --split train --output processed/unified_train.jsonl

Loaded 29195 span-answer examples from DROP.
Loaded 19399 examples from QUOREF.
Wrote 48594 examples to processed/unified_train.jsonl


In [ ]:
# bringing validation set into unified format
!python prepare_unified_data.py --drop data/drop_dataset_dev.json --quoref data/quoref-dev-v0.1.json --split validation --output processed/unified_validation.jsonl

Loaded 3529 span-answer examples from DROP.
Loaded 2418 examples from QUOREF.
Wrote 5947 examples to processed/unified_validation.jsonl


In [ ]:
# sanity check
!python controlled_experiments_and_further_analysis/run_sanity_overfit_official.py --base_train_file processed/unified_train.jsonl --drop_eval drop_eval.py --work_dir experiments/sanity_overfit --size 80 --epochs 30 --batch_size 8 --learning_rate 0.0003 --overwrite

In [ ]:
!python train_t5.py --train_file processed/unified_train.jsonl --validation_file processed/unified_validation.jsonl --output_dir models/t5_multispan --epochs 3 --batch_size 2 --gradient_accumulation_steps 4

In [ ]:
!python predict_t5.py --model_dir models/t5_multispan --input_file processed/unified_validation.jsonl --output_file predictions/validation_predictions.jsonl --batch_size 4

In [ ]:
!python evaluate_predictions.py --gold_file processed/unified_validation.jsonl --predictions_file predictions/validation_predictions.jsonl

In [ ]:
!python prepare_official_gold.py --drop_dev data/drop_dataset_dev.json --quoref_dev data/quoref-dev-v0.1.json --output_dir official_gold


In [ ]:
!python make_official_predictions.py --input_file predictions/validation_predictions.jsonl --gold_dir official_gold --output_dir official_predictions

In [ ]:

!python run_official_evaluation.py --drop_eval drop_eval.py --gold_dir official_gold --predictions_dir official_predictions --results_dir official_results


In [ ]:
# greedy decoding evaluation
!python controlled_experiments_and_further_analysis/run_controlled_experiment.py --experiment_name baseline_greedy --existing_model_dir models/t5_multispan --validation_file processed/unified_validation.jsonl --gold_dir official_gold --drop_eval drop_eval.py --make_official_script make_official_predictions.py --official_eval_script run_official_evaluation.py --num_beams 1 --work_dir experiments --overwrite

In [ ]:
# beam-4 decoding evaluation

!python controlled_experiments_and_further_analysis/run_controlled_experiment.py --experiment_name beam4 --existing_model_dir models/t5_multispan --validation_file processed/unified_validation.jsonl --gold_dir official_gold --drop_eval drop_eval.py --make_official_script make_official_predictions.py --official_eval_script run_official_evaluation.py --num_beams 4 --work_dir experiments --overwrite

In [ ]:
# generate-then-ground experiment

!python controlled_experiments_and_further_analysis/run_controlled_experiment.py --experiment_name grounded_filter --existing_model_dir models/t5_multispan --validation_file processed/unified_validation.jsonl --gold_dir official_gold --drop_eval drop_eval.py --make_official_script make_official_predictions.py --official_eval_script run_official_evaluation.py --num_beams 1 --postprocess_mode deduplicate_and_drop_ungrounded --work_dir experiments --overwrite

In [ ]:
# drop error analysis
!python controlled_experiments_and_further_analysis/analyze_official_errors.py --gold_file official_gold/drop_all_spans.json --predictions_file official_predictions/drop_predictions.json --drop_eval drop_eval.py --dataset_name DROP --output_dir analysis/errors/drop

python3: can't open file '/content/nlp_project/python': [Errno 2] No such file or directory


In [ ]:
# quoref error analysis
!python controlled_experiments_and_further_analysis/analyze_official_errors.py --gold_file official_gold/quoref_all_spans.json --predictions_file official_predictions/quoref_predictions.json --drop_eval drop_eval.py --dataset_name QUOREF --output_dir analysis/errors/quoref

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!zip -r /content/file.zip /content/nlp_project
from google.colab import files
files.download("/content/file.zip")


In [ ]:
import shutil

shutil.make_archive(
    base_name="/content/nlp_project",
    format="zip",
    root_dir="/content",
    base_dir="nlp_project",
)

print("Created: /content/nlp_project.zip")

In [ ]:
from google.colab import files

files.download("/content/nlp_project.zip")

In [ ]:
# here we transfer the nlp_project folder to my google drive
!cp -r "/content/nlp_project" "/content/drive/MyDrive/"